# Global Deforestation Analysis (1990–2025)
## Notebook 01: Data Collection, Schema Harmonization & Preprocessing Pipeline

### Scientific Overview & Research Context
Forests are the terrestrial biosphere's largest carbon sinks and terrestrial biodiversity reservoirs, covering approximately 31% of the global land surface. Between 1990 and 2025, anthropogenic pressures—principally agricultural expansion, commercial logging, mining, and climate-induced wildfire regimes—have profoundly reshaped global forest geography.

This notebook establishes an authentic, reproducible data pipeline. In strict compliance with scientific integrity principles, **no simulated or dummy data is utilized**. Every record is sourced from official, publicly available repositories:
1. **FAO Global Forest Resources Assessment (FRA 2020 / 2025)**: National reporting, satellite remote sensing, and official governmental forestry submissions.
2. **Global Forest Watch / Curtis et al. (Science 2018)**: High-resolution (30m Landsat) physical driver classification.
3. **Pendrill et al. (2022) / FAOSTAT**: Econometric attribution linking spatial clearing to international agricultural commodities.
4. **Global Carbon Project (GCP 2024)**: Biogeochemical bookkeeping modeling of CO₂ emissions from land-use change.
5. **IUCN Red List of Threatened Species**: Biodiversity integrity and extinction risk indices.


In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np

# Set project working directory
PROJ_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(PROJ_ROOT)

from src.utils import THEME, load_processed_data
print(f"Working Directory: {PROJ_ROOT}")


Working Directory: D:\DVLAB


### 1. Ingestion of Raw Authentic Datasets
We load the raw CSV files downloaded directly from official sources into memory and inspect their dimensions, column structures, and time horizons.


In [2]:
raw_dir = os.path.join(PROJ_ROOT, "data", "raw")

datasets_raw = {
    "Forest Area (ha)": "forest_area.csv",
    "Annual Net Change (ha/yr)": "annual_change_forest_area.csv",
    "Gross Deforestation (ha/yr)": "annual_deforestation.csv",
    "Gross Expansion (ha/yr)": "annual_forest_expansion.csv",
    "Drivers of Loss": "tree_cover_loss_by_dominant_driver.csv",
    "Commodity Attribution": "drivers_deforestation_commodity.csv",
    "Forest Share (% land)": "forest_area_share_land.csv",
    "Land-Use CO2 Emissions": "annual_co2_land_use.csv",
    "IUCN Red List Index": "red_list_index.csv"
}

summary_records = []
for name, fname in datasets_raw.items():
    fpath = os.path.join(raw_dir, fname)
    if os.path.exists(fpath):
        df_temp = pd.read_csv(fpath)
        summary_records.append({
            "Dataset": name,
            "Filename": fname,
            "Rows": len(df_temp),
            "Columns": len(df_temp.columns),
            "Min Year": df_temp["Year"].min() if "Year" in df_temp.columns else "N/A",
            "Max Year": df_temp["Year"].max() if "Year" in df_temp.columns else "N/A",
            "Unique Entities": df_temp["Entity"].nunique() if "Entity" in df_temp.columns else "N/A"
        })

df_raw_overview = pd.DataFrame(summary_records)
df_raw_overview


,Dataset,Filename,Rows,Columns,Min Year,Max Year,Unique Entities
0,Forest Area (ha),forest_area.csv,8928,4,1990,2025,248
1,Annual Net Change (ha/yr),annual_change_forest_area.csv,8680,4,1991,2025,248
2,Gross Deforestation (ha/yr),annual_deforestation.csv,705,4,1990,2020,151
3,Gross Expansion (ha/yr),annual_forest_expansion.csv,666,4,1990,2020,139
4,Drivers of Loss,tree_cover_loss_by_dominant_driver.csv,4750,10,2001,2024,209
5,Commodity Attribution,drivers_deforestation_commodity.csv,4001,11,2001,2023,192
6,Forest Share (% land),forest_area_share_land.csv,8078,5,1000,2025,224
7,Land-Use CO2 Emissions,annual_co2_land_use.csv,37450,4,1850,2024,214
8,IUCN Red List Index,red_list_index.csv,12013,4,1980,2025,262


### 2. ISO-3 Country Harmonization & Continental Assignment
Because datasets aggregate regional entities (e.g. `World`, `Europe`, `Low-income countries`) alongside sovereign states, we map standard ISO-3166-1 alpha-3 codes to standard UN continental regions:
- **Africa**
- **Asia**
- **Europe**
- **North America** (including Central America and Caribbean)
- **South America**
- **Oceania**


In [3]:
regions_path = os.path.join(raw_dir, "country_regions.csv")
df_geo = pd.read_csv(regions_path)
print(f"Total reference countries in ISO-3 mapping: {len(df_geo)}")
df_geo.head()


Total reference countries in ISO-3 mapping: 249


,alpha-3,name,region,sub-region,intermediate-region
0,AFG,Afghanistan,Asia,Southern Asia,NaN
1,ALA,Åland Islands,Europe,Northern Europe,NaN
2,ALB,Albania,Europe,Southern Europe,NaN
3,DZA,Algeria,Africa,Northern Africa,NaN
4,ASM,American Samoa,Oceania,Polynesia,NaN


### 3. Inspection of Processed Country Panel (1990–2025)
The processed panel consolidates all indicators by country and year. We inspect the cleaned panel structure and verify unit conversions:
$$1\text{ hectare (ha)} = 10,000\text{ m}^2 = 0.01\text{ km}^2$$
$$1\text{ km}^2 = 100\text{ ha}$$
$$1\text{ Million hectares (Mha)} = 1,000,000\text{ ha} = 10,000\text{ km}^2$$


In [4]:
data = load_processed_data()
df_panel = data["country_panel"]
print(f"Loaded Country Panel: {df_panel.shape[0]} rows across {df_panel['ISO3'].nunique()} countries.")
df_panel.sample(5, random_state=42)[["ISO3", "Entity", "Continent", "Year", "Forest_Area_Mha", "Annual_Change_Mha", "Forest_Share_pct", "Net_Change_Type"]]


Loaded Country Panel: 8460 rows across 235 countries.


,ISO3,Entity,Continent,Year,Forest_Area_Mha,Annual_Change_Mha,Forest_Share_pct,Net_Change_Type
6678,SJM,Svalbard and Jan Mayen,Europe,2008,0.000000,0.000000,NaN,Stable (Net Zero)
6656,SGP,Singapore,Asia,2022,0.015800,0.000000,22.005571,Stable (Net Zero)
625,BES,Bonaire Sint Eustatius and Saba,North America,2003,0.004210,0.000000,NaN,Stable (Net Zero)
4959,MMR,Myanmar,Asia,2017,29.413016,-0.289712,45.065678,Deforestation (Net Loss)
828,BIH,Bosnia and Herzegovina,Europe,1990,2.210000,NaN,43.164062,Unknown


### 4. Verification of Global Totals (FAO FRA 2020 / 2025)
Let us check the global total forest area and net change against published FAO statistics:
- **1990 Global Forest Area**: ~4.34 Billion ha (~43.4 Million km²)
- **2020 Global Forest Area**: ~4.17 Billion ha (~41.7 Million km²)
- **2025 Global Forest Area**: ~4.14 Billion ha (~41.4 Million km²)
- **Cumulative Global Net Loss (1990–2025)**: Over 200 Million hectares.


In [5]:
df_global = data["global_summary"]
df_global[["Year", "Forest_Area_Mha", "Annual_Change_Mha", "Forest_Share_pct", "Gross_Deforestation_Mha", "Gross_Expansion_Mha"]].dropna(how="all").tail(10)


,Year,Forest_Area_Mha,Annual_Change_Mha,Forest_Share_pct,Gross_Deforestation_Mha,Gross_Expansion_Mha
26,2016,4178.1942,-3.238765,32.198486,NaN,NaN
27,2017,4174.9555,-3.238765,32.173527,NaN,NaN
28,2018,4171.7166,-3.238765,32.148567,NaN,NaN
29,2019,4168.4780,-3.238765,32.123608,NaN,NaN
30,2020,4165.2393,-3.238765,32.098650,8.49011,5.34986
31,2021,4160.2342,-5.004931,32.060080,NaN,NaN
32,2022,4155.2292,-5.004931,32.021510,NaN,NaN
33,2023,4150.2244,-5.004931,31.982940,NaN,NaN
34,2024,4145.2193,-5.004931,31.944370,NaN,NaN
35,2025,4140.2145,-5.004931,31.905802,NaN,NaN


### Statistical Findings & Data Pipeline Conclusions
1. **High Quality Time-Series**: Harmonized time-series from 1990 to 2025 across 235 countries and territories provides seamless comparability.
2. **Gross vs Net Distinction**: Net annual change accounts for both destructive land-use conversion (gross loss) and natural regeneration / commercial plantations (gross expansion).
3. **Data Integrity**: Missing values are identified explicitly, maintaining strict compliance with peer-reviewed literature.
